# 知识图谱记忆（Knowledge Graph Memory）

> **每一轮对话都抽取关系三元组（主谓宾），用 NetworkX 构建内存中的图，再查询多跳路径，回答那些需要把散落在对话各处的事实连接起来的问题。**

想想家谱或组织架构图。这类图不会孤立地存放关于某个人的事实，而是展示人与人如何连接：「Alice 管理 Bob」「Bob 参与 Project X」「Project X 依赖 Q3 预算」。沿着这些连接追溯路径，就能回答任何单一事实都回答不了的问题。

前面介绍过的记忆技术，存储的是*关于*单个实体的信息（实体记忆），或者*关于*对话本身的信息（缓冲、摘要、向量库）。它们都不能以结构化、可遍历的方式捕捉**实体之间的关系**。

**知识图谱记忆**走的是另一条路。每轮对话之后，系统抽取三元组（`(Alice, manages, Bob)`、`(Project Alpha, depends_on, Q3 Budget)`）并加进一张有向图。当用户问「所有在 Alice 管理的项目上工作的人都有谁？」时，系统遍历这张图，跨多跳沿着边走，拼出一个任何单一事实都给不出的答案。

**读完本 notebook 你会理解：**
- 如何构建一个以 NetworkX 有向图为底座的三元组存储。
- 如何用 LLM 工具调用（tool calling）从对话中抽取关系三元组。
- 子图检索如何只把相关的关系上下文注入提示词。
- 多跳推理：回答跨越多个相互关联事实的问题。
- 在关系密集的回忆任务上，知识图谱记忆与更简单的做法相比表现如何。


## 关键概念

- **三元组（subject-predicate-object，主谓宾）：** 知识图谱的原子单位。每条关系都表达为一条有向边：`(主体) --[谓词]--> (客体)`。例如：`(Alice, manages, Bob)` 或 `(Wavelength, uses, Python)`。
- **知识图谱：** 一种有向、带标签的图。节点是实体，边是有类型的关系。和平铺的实体存储不同，图谱捕捉的是事物*彼此之间*如何关联。
- **三元组抽取：** 用 LLM（通过 tool use / function calling）把自由格式的对话文本解析成结构化三元组。比基于规则的方法更灵活，但噪声也更大。
- **子图检索：** 给定一个查询，先识别其中提到的实体，再从整张图中抽出它们的局部邻域（1-2 跳以内）。进入提示词的只有相关的子图。
- **多跳推理：** 回答需要跨越多条边的问题。例如「哪些项目依赖 Alice 掌控的预算？」需要沿 `Alice -> controls -> Budget -> depended_on_by -> Project` 一路走过去。
- **图谱持久化：** 把图序列化（比如存成 JSON 边列表）以跨会话保存。关系知识因此能在对话之间留存。


## Architecture

<p align="center">
  <img src="../../images/diagrams/08_knowledge_graph_memory.svg" alt="知识图谱记忆架构图" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart LR
    subgraph Turn["Each Conversation Turn"]
        A["User message"] --> B["Triple Extractor\n(tool calling)"]
        B --> C["Extracted triples\n(S, P, O)"]
        C --> D["Update\nKnowledge Graph"]
    end

    subgraph Graph["Knowledge Graph (NetworkX DiGraph)"]
        D --> E[("Nodes = entities\nEdges = predicates\n(Alice)--manages-->(Bob)")]
    end

    subgraph Response["Response Generation"]
        A --> F["Identify mentioned\nentities"]
        E --> F
        F --> G["Retrieve subgraph\n(1-2 hop neighborhood)"]
        G --> H["Build prompt:\nsystem + graph context\n+ recent messages"]
        H --> I["LLM\n(via deepagents)"]
        I --> J["Response"]
    end

    subgraph Persist["Persistence"]
        E --> K["Save edge list\n(JSON)"]
        K --> L["Load in\nnew session"]
        L --> E
    end

    style E fill:#4f46e5,color:#fff
    style I fill:#059669,color:#fff
    style B fill:#d97706,color:#fff
```

</details>

In [ ]:
# 安装所需依赖（运行一次）
%pip install -q deepagents python-dotenv matplotlib numpy networkx langchain-openai

加载环境变量并初始化模型。聊天走 deepagents 框架，三元组抽取用工具调用，图用 NetworkX 维护。模型是任意 OpenAI 兼容端点。


In [ ]:
import os
import json
from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

import networkx as nx
from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)


def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")
print(f"✓ networkx version: {nx.__version__}")

## 核心实现

我们的知识图谱记忆系统有三个组件：

1. **`TripleStore`：** 以 NetworkX `DiGraph` 为底座的存储。节点是实体，边带 `predicate` 标签。支持添加三元组、查询邻域、寻找路径和序列化。
2. **`TripleExtractor`：** 用 LLM 工具调用把自由文本解析成 `(subject, predicate, object)` 三元组。LLM 通过 function calling 充当关系解析器。
3. **`KnowledgeGraphMemory`：** 主类，负责统筹抽取、图更新、子图检索、上下文注入和聊天。


In [ ]:
class TripleStore:
    """基于 NetworkX 有向图的三元组存储。"""

    def __init__(self):
        self.graph = nx.DiGraph()

    def add_triple(self, subject: str, predicate: str, obj: str) -> None:
        """向图中添加一条 (subject, predicate, object) 三元组。"""
        s, o = subject.strip(), obj.strip()
        p = predicate.strip().lower().replace(" ", "_")

        self.graph.add_node(s)
        self.graph.add_node(o)

        # 已有完全相同的边则跳过；否则更新
        existing = self.graph.get_edge_data(s, o)
        if existing and existing.get("predicate") == p:
            return
        self.graph.add_edge(s, o, predicate=p)

    def get_triples(self) -> list[tuple[str, str, str]]:
        """以 (subject, predicate, object) 元组的形式返回所有三元组。"""
        return [
            (u, data["predicate"], v)
            for u, v, data in self.graph.edges(data=True)
        ]

    def get_entity_triples(self, entity: str) -> list[tuple[str, str, str]]:
        """获取与某个实体相关的全部三元组（作为主体或客体）。"""
        triples = []
        e = entity.strip()
        for _, v, data in self.graph.out_edges(e, data=True):
            triples.append((e, data["predicate"], v))
        for u, _, data in self.graph.in_edges(e, data=True):
            triples.append((u, data["predicate"], e))
        return triples

    def get_neighborhood(self, entities: list[str], depth: int = 2) -> list[tuple[str, str, str]]:
        """检索距给定实体 `depth` 跳以内的子图。"""
        relevant_nodes = set()
        for entity in entities:
            e = entity.strip()
            if e not in self.graph:
                continue
            undirected = self.graph.to_undirected()
            reachable = nx.single_source_shortest_path_length(undirected, e, cutoff=depth)
            relevant_nodes.update(reachable.keys())

        triples = []
        for u, v, data in self.graph.edges(data=True):
            if u in relevant_nodes or v in relevant_nodes:
                triples.append((u, data["predicate"], v))
        return triples



接下来添加图查询方法。`find_path` 用 NetworkX 的最短路径算法在图中追溯两个实体之间的路线。`find_mentioned` 在文本中扫描已知实体名。`format_triples` 把三元组转成可读文本行。

这里用了 notebook 常用的分块定义技巧：先定义函数，再把它们挂到 `TripleStore` 上。Python 的类是可变对象，可以随时添加方法。这样每个代码单元格都保持完整、独立可执行。


In [ ]:
def _find_path(self, source: str, target: str) -> list[tuple[str, str, str]] | None:
    """找出两个实体之间的路径，以三元组列表返回。"""
    s, t = source.strip(), target.strip()
    undirected = self.graph.to_undirected()
    try:
        path_nodes = nx.shortest_path(undirected, s, t)
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return None

    triples = []
    for i in range(len(path_nodes) - 1):
        u, v = path_nodes[i], path_nodes[i + 1]
        if self.graph.has_edge(u, v):
            p = self.graph[u][v]["predicate"]
            triples.append((u, p, v))
        elif self.graph.has_edge(v, u):
            p = self.graph[v][u]["predicate"]
            triples.append((v, p, u))
    return triples


def _find_mentioned(self, text: str) -> list[str]:
    """找出文本中提到的所有图内实体。"""
    text_lower = text.lower()
    return [node for node in self.graph.nodes if node.lower() in text_lower]


def _format_triples(self, triples: list[tuple[str, str, str]]) -> str:
    """把三元组格式化成可读文本，用于注入提示词。"""
    if not triples:
        return "No known relationships."
    seen = set()
    lines = []
    for s, p, o in triples:
        if (s, p, o) not in seen:
            seen.add((s, p, o))
            lines.append(f"  {s} --[{p}]--> {o}")
    return "\n".join(lines)


# 把方法挂到 TripleStore 上（类是可变对象，可随时添加方法）
TripleStore.find_path = _find_path
TripleStore.find_mentioned = _find_mentioned
TripleStore.format_triples = _format_triples

print("✓ Graph query methods attached")

这里添加持久化方法。`save` 方法把图写成 JSON 边列表。`load` 类方法从该文件重建图。这样智能体的关系知识就能跨会话留存。沿用同样的挂载技巧。


In [ ]:
def _save(self, path: str) -> None:
    """把图持久化为 JSON 边列表。"""
    data = {
        "nodes": list(self.graph.nodes),
        "edges": [
            {"subject": u, "predicate": d["predicate"], "object": v}
            for u, v, d in self.graph.edges(data=True)
        ],
    }
    with open(path, "w") as f:
        json.dump(data, f, indent=2)


@classmethod
def _load(cls, path: str) -> "TripleStore":
    """从 JSON 边列表加载图。"""
    store = cls()
    with open(path) as f:
        data = json.load(f)
    for edge in data["edges"]:
        store.add_triple(edge["subject"], edge["predicate"], edge["object"])
    return store


def _len(self) -> int:
    return self.graph.number_of_edges()


def _repr(self) -> str:
    return f"TripleStore({self.graph.number_of_nodes()} nodes, {self.graph.number_of_edges()} edges)"


TripleStore.save = _save
TripleStore.load = _load
TripleStore.__len__ = _len
TripleStore.__repr__ = _repr

print(f"✓ TripleStore ready: {TripleStore()}")

我们定义三元组抽取的工具 schema。这份 JSON schema 遵循 OpenAI 工具调用格式，告诉模型如何调用我们的 `store_triples` 函数。它规定每条三元组必须包含三个字段：subject（主体）、predicate（关系类型）、object（客体）。


In [ ]:
# 定义三元组抽取的工具 schema（OpenAI 工具调用格式）
EXTRACT_TRIPLES_TOOL = {
    "type": "function",
    "function": {
        "name": "store_triples",
        "description": (
            "Extract relationship triples (subject-predicate-object) from the conversation. "
            "Each triple represents a relationship between two entities. "
            "Call this tool with ALL relationships mentioned or implied."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "triples": {
                    "type": "array",
                    "description": "List of relationship triples extracted from the message.",
                    "items": {
                        "type": "object",
                        "properties": {
                            "subject": {
                                "type": "string",
                                "description": "The source entity (e.g., 'Alice', 'Project Alpha').",
                            },
                            "predicate": {
                                "type": "string",
                                "description": "The relationship type (e.g., 'manages', 'works_at', 'depends_on').",
                            },
                            "object": {
                                "type": "string",
                                "description": "The target entity (e.g., 'Bob', 'Acme Corp').",
                            },
                        },
                        "required": ["subject", "predicate", "object"],
                    },
                },
            },
            "required": ["triples"],
        },
    },
}

`TripleExtractor` 把每一轮对话连同上面的工具 schema 发给模型。`tool_choice="required"` 强制模型必须发起一次工具调用，等价于 Anthropic 的 tool forcing。模型解析文本并返回结构化三元组，LangChain 会把工具调用解析进 `response.tool_calls`。我们使用 snake_case 风格的谓词（如 `works_at`、`manages`）以保证一致性。


In [ ]:
class TripleExtractor:
    """用工具调用从消息中抽取关系三元组。"""

    def __init__(self, model):
        # tool_choice="required" 强制模型必须调用工具
        self.model = model.bind_tools([EXTRACT_TRIPLES_TOOL], tool_choice="required")

    def extract(self, user_msg: str, assistant_msg: str) -> list[dict]:
        """从一轮用户-助手对话中抽取三元组。

        返回字典组成的列表，键为：subject、predicate、object。
        """
        response = self.model.invoke([
            {"role": "system", "content": (
                "You are a relationship extraction system. Analyze the conversation "
                "and extract ALL relationships as (subject, predicate, object) triples. "
                "Use concise, consistent entity names (e.g., 'Alice' not 'my friend Alice'). "
                "Use snake_case predicates (e.g., 'works_at', 'manages', 'located_in'). "
                "If no relationships are found, call store_triples with an empty list."
            )},
            {"role": "user", "content": (
                f"Extract relationship triples from this exchange:\n\n"
                f"User: {user_msg}\n"
                f"Assistant: {assistant_msg}"
            )},
        ])

        for call in response.tool_calls or []:
            if call["name"] == "store_triples":
                return call["args"].get("triples", [])
        return []


print("✓ TripleExtractor class defined (uses tool calling)")

`KnowledgeGraphMemory` 类负责统筹一切。构建提示词时，它会找出用户消息中提到的实体，取回它们在图中的邻域（2 跳以内的全部三元组），并把这份子图作为上下文注入。聊天走 deepagents，三元组抽取走独立的工具调用，两者共用同一个模型实例。


In [ ]:
class KnowledgeGraphMemory:
    """带知识图谱记忆的聊天智能体：抽取三元组、构建图，
    并把相关子图上下文注入每条提示词。"""

    def __init__(
        self,
        model,
        system_prompt: str | None = None,
        recent_buffer_size: int = 10,
        neighborhood_depth: int = 2,
        triple_store: TripleStore | None = None,
    ):
        self.model = model
        self.base_system_prompt = system_prompt or "You are a helpful assistant."
        self.recent_buffer_size = recent_buffer_size
        self.neighborhood_depth = neighborhood_depth

        # 组件
        self.store = triple_store or TripleStore()
        self.extractor = TripleExtractor(model=model)

        # 状态
        self.messages: list[dict] = []
        self.turn_count = 0

    def _build_system_prompt(self, user_input: str) -> str:
        """把相关的图上下文注入系统提示词。"""
        parts = [self.base_system_prompt]

        # 找出当前消息中提到的实体
        mentioned = self.store.find_mentioned(user_input)
        if mentioned:
            neighborhood = self.store.get_neighborhood(
                mentioned, depth=self.neighborhood_depth
            )
            if neighborhood:
                graph_context = self.store.format_triples(neighborhood)
                parts.append(
                    f"You have the following relationship knowledge relevant "
                    f"to the user's message:\n\n{graph_context}\n\n"
                    f"Use these relationships to give informed, accurate responses. "
                    f"You can follow chains of relationships to answer multi-hop questions."
                )

        # 整张图的概览
        all_triples = self.store.get_triples()
        if all_triples:
            entities = sorted(set(
                e for s, p, o in all_triples for e in (s, o)
            ))
            parts.append(f"All known entities: {', '.join(entities)}")

        return "\n\n".join(parts)

`chat` 方法是主循环。它先构建带图上下文的系统提示词，创建一个 deepagents 智能体（图在增长，系统提示词每轮都不同，所以每轮重建），把最近的消息窗口交给它，然后从这轮对话中抽取新三元组并加入图。


In [ ]:
def _chat(self, user_input: str) -> str:
    """发送消息、抽取三元组、更新图，并带上下文回复。"""
    self.turn_count += 1

    system = self._build_system_prompt(user_input)

    self.messages.append({"role": "user", "content": user_input})
    recent = self.messages[-self.recent_buffer_size:]

    # deepagents 的 invoke 无状态：系统提示词带最新图上下文，每轮重建智能体
    agent = create_deep_agent(model=self.model, system_prompt=system)
    result = agent.invoke({"messages": recent})
    assistant_text = to_text(result["messages"][-1].content)

    self.messages.append({"role": "assistant", "content": assistant_text})

    # 从这轮对话抽取三元组
    try:
        extracted = self.extractor.extract(user_input, assistant_text)
        for triple in extracted:
            self.store.add_triple(
                triple["subject"], triple["predicate"], triple["object"]
            )
    except Exception as e:
        print(f"  [extraction warning: {e}]")

    return assistant_text


KnowledgeGraphMemory.chat = _chat

print("✓ chat method attached")

这里添加实用方法。`query_graph` 返回单个实体的所有关系。`find_connection` 追溯两个实体之间的最短路径。`save_memory` 和 `from_saved` 负责跨会话持久化。同样以挂载方式加到类上。


In [ ]:
def _query_graph(self, entity: str) -> str:
    """查询某个实体的所有已知关系。"""
    triples = self.store.get_entity_triples(entity)
    if not triples:
        return f"No known relationships for '{entity}'."
    return self.store.format_triples(triples)


def _find_connection(self, entity1: str, entity2: str) -> str:
    """找出两个实体之间的最短路径。"""
    path = self.store.find_path(entity1, entity2)
    if path is None:
        return f"No connection found between '{entity1}' and '{entity2}'."
    return self.store.format_triples(path)


def _save_memory(self, path: str) -> None:
    """把知识图谱持久化到磁盘。"""
    self.store.save(path)


@classmethod
def _from_saved(cls, path: str, **kwargs) -> "KnowledgeGraphMemory":
    """创建实例并预加载已保存的知识图谱。"""
    store = TripleStore.load(path)
    return cls(triple_store=store, **kwargs)


def _repr(self) -> str:
    return (
        f"KnowledgeGraphMemory(turns={self.turn_count}, "
        f"nodes={self.store.graph.number_of_nodes()}, "
        f"edges={self.store.graph.number_of_edges()})"
    )


KnowledgeGraphMemory.query_graph = _query_graph
KnowledgeGraphMemory.find_connection = _find_connection
KnowledgeGraphMemory.save_memory = _save_memory
KnowledgeGraphMemory.from_saved = _from_saved
KnowledgeGraphMemory.__repr__ = _repr

print("✓ KnowledgeGraphMemory complete")

## 使用示例：构建关系图

我们来进行一场多轮对话，在人物、项目和地点之间建立一张关系网。每轮之后，系统抽取三元组，让知识图谱不断生长。后面的轮次就能受益于基于图的上下文注入。


In [ ]:
mem = KnowledgeGraphMemory(
    model=llm,
    system_prompt="You are a friendly personal assistant. Reply concisely in 1-2 sentences.",
)

messages = [
    "Hi! I'm Priya. I work as a data scientist at Spotify.",
    "My manager is Carlos -- he leads the recommendations team.",
    "Carlos also manages Aisha, who's a machine learning engineer.",
    "I'm working on Project Wavelength with Aisha. It's a music recommendation engine.",
    "Wavelength depends on a dataset called MusicGraph that the data engineering team maintains.",
    "My partner Alex is a chef -- he runs a restaurant called Basil & Lime in Portland.",
    "How are Aisha and Carlos related?",
    "What do you know about Project Wavelength and its dependencies?",
]

for msg in messages:
    print(f"👤 User:  {msg}")
    reply = mem.chat(msg)
    print(f"🤖 Agent: {reply}")
    print()

print(f"\n📊 Graph: {mem.store.graph.number_of_nodes()} nodes, {mem.store.graph.number_of_edges()} edges")

我们来检查这张知识图谱：打印每一条三元组（关系），列出所有已知实体，查询特定实体的连接，再追溯两个实体之间的路径。


In [ ]:
# 检查知识图谱
print("=== All Triples in the Knowledge Graph ===\n")
for s, p, o in mem.store.get_triples():
    print(f"  ({s}) --[{p}]--> ({o})")

print(f"\n=== Entities: {sorted(mem.store.graph.nodes)} ===")

# 查询特定实体的关系
print("\n=== Relationships for 'Carlos' ===")
print(mem.query_graph("Carlos"))

# 查找实体之间的连接
print("\n=== Path: Priya \u2192 MusicGraph ===")
print(mem.find_connection("Priya", "MusicGraph"))

## 可视化知识图谱

基于图的记忆有一个优势：智能体学到了哪些关系，可以直接画出来看。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(12, 7))

G = mem.store.graph
pos = nx.spring_layout(G, seed=42, k=2.5)

# 画节点
nx.draw_networkx_nodes(G, pos, ax=ax, node_color="#4f46e5", node_size=1800, alpha=0.9)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=9, font_color="white", font_weight="bold")

# 画边和边标签
edge_labels = {(u, v): d["predicate"] for u, v, d in G.edges(data=True)}
nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#888", arrows=True,
                       arrowsize=20, connectionstyle="arc3,rad=0.1", width=1.5)
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax,
                             font_size=7, font_color="#d97706")

ax.set_title("Knowledge Graph: Extracted Relationships", fontsize=14, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.savefig("knowledge_graph.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\u2713 Graph visualization saved (knowledge_graph.png)")

## 多跳推理

这是图记忆相对实体记忆的关键优势：回答需要**跨越多条关系**的问题。智能体沿着边组成的链条，把出现在完全不同轮次里的事实连接起来。


In [ ]:
# 需要连接多条三元组的多跳问题
multi_hop_questions = [
    "What dataset does the project I'm working on depend on?",
    "Who else does my manager manage besides me?",
    "What's the connection between Alex and Spotify?",
]

print("=== Multi-Hop Reasoning ===\n")
for question in multi_hop_questions:
    print(f"\U0001f464 User:  {question}")
    reply = mem.chat(question)
    print(f"\U0001f916 Agent: {reply}")
    print()

## 跨会话持久化

知识图谱可以序列化成 JSON 边列表。新会话加载这张图，立刻就能访问之前学到的所有关系。用户再也不用重新介绍自己的世界。


In [ ]:
# -- 保存会话 1 的知识图谱 --
KG_FILE = "knowledge_graph.json"
mem.save_memory(KG_FILE)
print(f"✓ Saved graph ({len(mem.store)} triples) to {KG_FILE}")

# -- 模拟新会话（会话 2）--
print("\n--- NEW SESSION ---\n")

mem2 = KnowledgeGraphMemory.from_saved(
    KG_FILE,
    model=llm,
    system_prompt="You are a friendly personal assistant. Reply concisely in 1-2 sentences.",
)
print(f"Loaded graph: {mem2.store.graph.number_of_nodes()} nodes, {mem2.store.graph.number_of_edges()} edges")
print(f"Known entities: {sorted(mem2.store.graph.nodes)}\n")

session2_questions = [
    "Who manages the recommendations team?",
    "What project am I working on, and who's on it with me?",
    "Tell me about Alex.",
]

for msg in session2_questions:
    print(f"👤 User:  {msg}")
    reply = mem2.chat(msg)
    print(f"🤖 Agent: {reply}")
    print()

print("✓ Cross-session persistence works -- relationships preserved!")

## 实验：知识图谱记忆 vs. 滑动窗口

我们来做一个受控测试，专注于**关系密集的回忆**：

1. 在一场 **30 轮对话**中埋入 **10 条关系事实**（混在闲聊里）。
2. 提出 **10 个回忆问题**，其中多个需要多跳推理。
3. 对比**知识图谱记忆**（抽取并存储三元组）和**滑动窗口**（只保留最近 10 条消息）。

假设是：知识图谱记忆会在关系题和多跳题上表现出色，而滑动窗口对付已经滑出窗口的事实会很吃力。


In [ ]:
# 在特定轮次埋入的关系事实
PLANTED_FACTS = {
    1:  ("My name is Jordan and I'm a software engineer at Nexus Corp.",         "jordan",       "engineer"),
    3:  ("My team lead is Sana -- she also mentors our intern, Leo.",             "sana",         "team_lead"),
    5:  ("Leo is working on Project Beacon, which is a real-time analytics platform.", "beacon",  "project"),
    8:  ("Project Beacon depends on a service called DataStream that Raj built.",  "datastream",   "dependency"),
    11: ("Raj works in the infrastructure team, which Sana used to lead.",         "raj",          "infra"),
    15: ("My partner Morgan is a teacher at Oakwood Elementary.",                  "morgan",       "partner"),
    18: ("Morgan coaches the school's debate team with a colleague named Dana.",   "dana",         "colleague"),
    22: ("Dana is also Leo's older sister -- small world!",                        "dana",         "family"),
    26: ("I'm thinking of taking a course on graph databases taught by Prof. Chen.", "chen",       "professor"),
    29: ("Prof. Chen wrote the textbook we use at Nexus Corp for onboarding.",      "chen",        "textbook"),
}

FILLER_MESSAGES = [
    "What's the difference between HTTP and HTTPS?",
    "Can you explain quantum entanglement simply?",
    "What's a good recipe for banana bread?",
    "How does a heat pump work?",
    "Tell me about the history of jazz.",
    "What's the tallest mountain in South America?",
    "How do noise-canceling headphones work?",
    "What causes auroras?",
    "Explain the Pythagorean theorem.",
    "What's the oldest known civilization?",
    "How does sourdough starter work?",
    "What is the Doppler effect?",
    "Tell me about coral reefs.",
    "How does memory work in the brain?",
    "What's the speed of light?",
    "How do electric cars regenerate braking energy?",
    "What is dark energy?",
    "Tell me about the Silk Road trade routes.",
    "How do birds navigate during migration?",
    "What is the Krebs cycle?",
]



我们把埋入的关系事实和闲聊消息交错，构成 30 轮对话。然后定义回忆问题。其中几个问题需要多跳推理：通过图把不同轮次的事实连接起来。


In [ ]:
# 构建 30 轮对话
conversation_30 = []
filler_idx = 0
for turn in range(1, 31):
    if turn in PLANTED_FACTS:
        msg = PLANTED_FACTS[turn][0]
    else:
        msg = FILLER_MESSAGES[filler_idx % len(FILLER_MESSAGES)]
        filler_idx += 1
    conversation_30.append((turn, msg))

# 有些问题需要多跳图遍历
RECALL_QUESTIONS = [
    ("Where do I work?",                                          "nexus",       1),
    ("Who is my team lead?",                                      "sana",        3),
    ("What project is Leo working on?",                           "beacon",      5),
    ("What does Project Beacon depend on?",                       "datastream",  8),
    ("What team does Raj work on?",                               "infrastructure", 11),
    ("Who is Morgan?",                                            "teacher",     15),
    ("Who does Morgan coach the debate team with?",               "dana",        18),
    ("How are Dana and Leo related?",                             "sister",      22),
    ("Who teaches the graph databases course?",                   "chen",        26),
    ("Who wrote the textbook used for onboarding at Nexus Corp?", "chen",        29),
]

print(f"Conversation: {len(conversation_30)} turns")
print(f"Facts planted at turns: {sorted(PLANTED_FACTS.keys())}")
print(f"Recall questions: {len(RECALL_QUESTIONS)}")

我们把全部 30 轮对话跑一遍知识图谱记忆。每轮之后，抽取器识别关系三元组并加入图。看节点数和边数如何增长。


In [ ]:
# -- 30 轮对话跑知识图谱记忆 --

kg_mem = KnowledgeGraphMemory(
    model=llm,
    system_prompt="You are a helpful assistant. Reply concisely in 1-2 sentences.",
)

print("Running 30-turn conversation through Knowledge Graph Memory...")
for turn_num, msg in conversation_30:
    kg_mem.chat(msg)
    if turn_num % 10 == 0:
        print(f"  Turn {turn_num}/30 done ({kg_mem.store.graph.number_of_nodes()} nodes, {len(kg_mem.store)} edges)")

print(f"\n✓ Done. Graph: {kg_mem.store.graph.number_of_nodes()} nodes, {len(kg_mem.store)} edges")
print(f"\n=== Extracted Triples ===")
for s, p, o in kg_mem.store.get_triples():
    print(f"  ({s}) --[{p}]--> ({o})")

我们定义一个滑动窗口基线，把同样的 30 轮对话跑一遍。窗口只保留最近 10 条消息，较早的关系事实会掉出上下文。


In [ ]:
class SlidingWindowBaseline:
    """滑动窗口：只保留最近 K 条消息。"""

    def __init__(self, window_size: int = 10, model=None,
                 system_prompt: str | None = None):
        self.window_size = window_size
        self.model = model
        self.system_prompt = system_prompt or "You are a helpful assistant."
        self.messages: list[dict] = []

    def chat(self, user_input: str) -> str:
        self.messages.append({"role": "user", "content": user_input})
        window = self.messages[-self.window_size:]

        # 与 01 相同的教训：deepagents 的 invoke 无状态，窗口就是它仅存的记忆
        agent = create_deep_agent(model=self.model, system_prompt=self.system_prompt)
        result = agent.invoke({"messages": window})
        assistant_text = to_text(result["messages"][-1].content)

        self.messages.append({"role": "assistant", "content": assistant_text})
        return assistant_text


# -- 30 轮对话跑滑动窗口 --

sw_mem = SlidingWindowBaseline(
    window_size=10,
    model=llm,
    system_prompt="You are a helpful assistant. Reply concisely in 1-2 sentences.",
)

print("Running 30-turn conversation through Sliding Window Memory...")
for turn_num, msg in conversation_30:
    sw_mem.chat(msg)
    if turn_num % 10 == 0:
        print(f"  Turn {turn_num}/30 done")

print(f"\n✓ Done. Window keeps last {sw_mem.window_size} messages.")

我们对两套系统运行回忆测试。对每个问题，检查答案是否包含预期的关键词。在滑动窗口早已忘掉相关事实的多跳题上，知识图谱记忆应该会表现更好。


In [ ]:
# -- 回忆测试：知识图谱记忆 --
print("=== Knowledge Graph Memory - Recall Test ===\n")
kg_results = []

for question, keyword, planted_at in RECALL_QUESTIONS:
    answer = kg_mem.chat(question)
    recalled = keyword.lower() in answer.lower()
    kg_results.append({
        "question": question, "keyword": keyword,
        "planted_at": planted_at, "recalled": recalled,
        "answer": answer,
    })
    status = "\u2713" if recalled else "\u2717"
    print(f"  {status} (turn {planted_at:2d}) {question}")
    print(f"    Answer: {answer[:120]}")
    print()

# -- 回忆测试：滑动窗口 --
print("=== Sliding Window - Recall Test ===\n")
sw_results = []

for question, keyword, planted_at in RECALL_QUESTIONS:
    answer = sw_mem.chat(question)
    recalled = keyword.lower() in answer.lower()
    sw_results.append({
        "question": question, "keyword": keyword,
        "planted_at": planted_at, "recalled": recalled,
        "answer": answer,
    })
    status = "\u2713" if recalled else "\u2717"
    print(f"  {status} (turn {planted_at:2d}) {question}")
    print(f"    Answer: {answer[:120]}")
    print()

kg_score = sum(1 for r in kg_results if r["recalled"])
sw_score = sum(1 for r in sw_results if r["recalled"])
print(f"Knowledge Graph Memory score:  {kg_score}/{len(RECALL_QUESTIONS)}")
print(f"Sliding Window score:          {sw_score}/{len(RECALL_QUESTIONS)}")

我们来可视化这场对比。第 1 面板是回忆总分。第 2 面板按事实的新旧分解回忆情况。


In [ ]:
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# -- 第 1 面板：回忆总分 --
kg_score = sum(1 for r in kg_results if r["recalled"])
sw_score = sum(1 for r in sw_results if r["recalled"])
labels = ["Knowledge Graph\nMemory", "Sliding Window\n(K=10)"]
scores = [kg_score, sw_score]
colors = ["#4f46e5", "#ef4444"]

bars = axes[0].bar(labels, scores, color=colors, width=0.5, alpha=0.85)
axes[0].set_ylabel("Facts Recalled")
axes[0].set_ylim(0, len(RECALL_QUESTIONS) + 1)
axes[0].set_title(f"Total Facts Recalled (out of {len(RECALL_QUESTIONS)})")
axes[0].axhline(y=len(RECALL_QUESTIONS), color="gray", linestyle="--", alpha=0.3)
for bar, score in zip(bars, scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(score), ha="center", fontweight="bold", fontsize=14)

# -- 第 2 面板：按事实埋入时间分解 --
fact_turns = [r["planted_at"] for r in kg_results]
kg_recalled = [1 if r["recalled"] else 0 for r in kg_results]
sw_recalled = [1 if r["recalled"] else 0 for r in sw_results]

x = np.arange(len(fact_turns))
width = 0.35
axes[1].bar(x - width/2, kg_recalled, width, label="KG Memory", color="#4f46e5", alpha=0.85)
axes[1].bar(x + width/2, sw_recalled, width, label="Sliding Window", color="#ef4444", alpha=0.85)
axes[1].set_xlabel("Fact Planted at Turn #")
axes[1].set_ylabel("Recalled? (1=Yes, 0=No)")
axes[1].set_title("Recall by Fact Age")
axes[1].set_xticks(x)
axes[1].set_xticklabels([str(t) for t in fact_turns], fontsize=9)
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(["No", "Yes"])
axes[1].legend()

plt.tight_layout()
plt.savefig("kg_vs_window.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nKnowledge Graph Memory:  {kg_score}/{len(RECALL_QUESTIONS)} recalled")
print(f"Sliding Window:         {sw_score}/{len(RECALL_QUESTIONS)} recalled")

## 讨论与权衡

### 优势
- **关系优先。** 实体记忆存储*关于*实体的事实，知识图谱记忆捕捉的是实体*之间*的事实。这让它能做其他方法做不到的多跳推理。
- **可遍历的结构。** 你可以用程序查询这张图：最短路径、连通分量、邻域检索。不需要嵌入相似度。
- **可视化、可解释。** 图可以直接画出来。智能体「知道」什么，一眼就能检查和调试。
- **可与其他技术组合。** 知识图谱可以和滑动窗口（最近上下文）、摘要记忆（高层回顾）搭配使用。

### 劣势
- **抽取噪声。** 基于 LLM 的三元组抽取并不完美。谓词可能不一致（`manages` vs. `is_manager_of`），实体名可能有变体（`Professor Chen` vs. `Prof. Chen`），闲聊里也可能冒出虚假三元组。
- **双倍 API 开销。** 和实体记忆一样，每轮都要为抽取多付一次 LLM 调用，成本大约翻倍。
- **图维护。** 去重、规范化、冲突解决、修剪都需要持续投入。生产系统还需要实体消解和谓词规范化（把关系标签标准化）。
- **序列化开销。** 把图上下文塞进提示词，token 效率不如摘要记忆。图大了就得激进地过滤子图。
- **扩展性。** NetworkX 在内存里跑。生产环境有数百万条三元组时，请改用 Neo4j 或类似的图数据库。

### 知识图谱记忆 vs. 实体记忆

| 维度 | 实体记忆 | 知识图谱记忆 |
|--------|--------------|----------------------|
| 存什么 | *关于*实体的事实 | 实体*之间*的关系 |
| 结构 | 键值对（平铺） | 图（节点 + 边） |
| 查询方式 | 按名字查找 | 遍历、路径、邻域 |
| 多跳推理 | 不支持 | 核心优势 |
| 可视化 | 实体列表 | 图示 |
| 复杂度 | 更简单 | 更复杂 |
| 最适合 | 个人事实、偏好 | 组织结构、依赖关系 |

### 何时使用知识图谱记忆

| 场景 | 建议 |
|----------|---------------|
| 跟踪复杂关系（组织架构、项目依赖） | 非常合适，这就是它的核心用例 |
| 跨越多条事实的多跳问题 | 非常合适 |
| 只涉及个人事实的个人助理 | 杀鸡用牛刀，实体记忆足够了 |
| 需要扩展到数百万条关系 | 改用 Neo4j，别用 NetworkX |
| 对成本敏感的部署 | 谨慎：抽取会让每轮成本翻倍 |
| 与其他记忆类型组合 | 与实体记忆 + 滑动窗口是绝佳搭档 |


## 延伸阅读

- [OpenAI: Function Calling](https://platform.openai.com/docs/guides/function-calling?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：本 notebook 用来做三元组抽取的工具调用 API
- [deepagents GitHub 仓库](https://github.com/langchain-ai/deepagents)：本 notebook 的聊天框架
- [NetworkX: Python Graph Library](https://networkx.org/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：本 notebook 使用的内存图后端
- [Neo4j Graph Database](https://neo4j.com/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：超越内存方案、面向规模化的生产级图数据库
- [Microsoft GraphRAG](https://github.com/microsoft/graphrag)：规模化的基于图的检索增强生成
- [LangChain ConversationKGMemory](https://python.langchain.com/docs/modules/memory/types/kg?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：框架级的知识图谱记忆
- [Ji et al., "A Survey on Knowledge Graphs" (2022)](https://ieeexplore.ieee.org/document/9416312?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：关于知识图谱表示、获取与应用的综述

---

*← 上一篇：[07 - 实体记忆](../07_entity_memory/) · 下一篇：[09 - 情景记忆](../09_episodic_memory/) →*

In [ ]:
# 清理临时文件
import os
for f in ["knowledge_graph.json", "knowledge_graph.png", "kg_vs_window.png"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")

## 🧪 动手试一试

三个小挑战，加深理解。每个大约需要 10-30 分钟。

### 挑战 1：带置信度的三元组
给 `TripleStore` 里存储的每条三元组加一个 `confidence` 浮点数（0.0 到 1.0）。修改 `get_neighborhood()`，接受一个 `min_confidence` 参数，过滤掉低置信度的边。用一段用户纠正过先前说法的对话来测试。

### 挑战 2：图结构指标
30 轮实验结束后，计算并打印：总节点数、总边数、图密度（`nx.density()`）、连通分量个数，以及最大连通分量内的平均最短路径长度。讨论这些数字揭示了这场对话的什么特点。

### 挑战 3：时序边过期
给每条三元组加一个 `created_at` 时间戳。实现一个 `prune_stale()` 方法，删除超过可配置阈值的旧三元组。跑 30 轮，剪掉早于最近 10 轮的边，对比检索质量。这是把 18 时序记忆的思路扩展到图结构上。


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--08-knowledge-graph-memory--knowledge-graph-memory)
